In [122]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

In [123]:
df = pd.read_csv("../data/engineered_features.csv")

In [124]:
df.select_dtypes(include = ['str'])

,shipment_id,order_date,ship_date,shipment_mode,carrier,service_level,origin_metro,dest_metro
0,SHP-42-0003885,2024-04-23,2024-04-23,parcel,Cobalt Regional,Overnight,Los Angeles CA,Salt Lake City UT
1,SHP-42-0010239,2025-01-07,2025-01-09,parcel,Northstar Parcel,Expedited,Seattle WA,New York NY
2,SHP-42-0035486,2025-07-30,2025-07-30,ltl_freight,Vela Freight,LTL Standard,Atlanta GA,Miami FL
3,SHP-42-0008180,2025-03-24,2025-03-26,parcel,Northstar Parcel,Ground,New York NY,Salt Lake City UT
4,SHP-42-0015190,2025-05-06,2025-05-08,parcel,Falcon Post,Expedited,Los Angeles CA,Columbus OH
...,...,...,...,...,...,...,...,...
39742,SHP-42-0006265,2024-09-18,2024-09-20,parcel,Northstar Parcel,Ground,Nashville TN,Boston MA
39743,SHP-42-0011284,2024-09-23,2024-09-23,parcel,Northstar Parcel,TwoDay,Portland OR,Charlotte NC
39744,SHP-42-0044732,2025-05-30,2025-05-31,ltl_freight,Ironclad Freight,LTL Standard,Charlotte NC,Denver CO
39745,SHP-42-0038158,2024-11-22,2024-11-23,parcel,Summit Express,Ground,Phoenix AZ,Los Angeles CA


In [125]:
df["carrier"] = df["carrier"].astype("category")
df["service_level"] = df["service_level"].astype("category")
df["origin_metro"] = df["origin_metro"].astype("category")
df["dest_metro"] = df["dest_metro"].astype("category")
df["shipment_mode"] = df["shipment_mode"].astype("category")

In [126]:
y = df["transit_days"]
X = df.drop(columns = ["transit_days", "delivered_on_time", "shipment_id", "order_date", "ship_date", "origin_zip", "dest_zip"])


In [127]:
X_trainval, X_test_final, y_trainval, y_test_final = train_test_split(X, y, test_size=0.15, random_state = 1234)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.15/0.85, random_state=1234)
model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03, random_state=1234)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50)])
test_predictions = model.predict(X_test_final)
value_predictions = model.predict(X_val)

c:\Users\nabil\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014481 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2637
[LightGBM] [Info] Number of data points in the train set: 27822, number of used features: 30
[LightGBM] [Info] Start training from score 5.308425
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[298]	valid_0's l2: 0.603002


In [128]:
mean_score_val = mean_absolute_error(y_val, value_predictions)
mean_score_test = mean_absolute_error(y_test_final, test_predictions)
print("Mean Absolute Error- Validation Score: " + str(mean_score_val))
print("Mean Absolute Error- Test Score: " + str(mean_score_test))

Mean Absolute Error- Validation Score: 0.5460939155101339
Mean Absolute Error- Test Score: 0.539225779740578


In [129]:
baseline_mae_val = (y_val - X_val["promised_days"]).abs().mean()
baseline_mae_test = (y_test_final - X_test_final["promised_days"]).abs().mean()
print("Mean Absolute Error- Validation Baseline: " + str(baseline_mae_val))
print("Mean Absolute Error- Test Baseline: " + str(baseline_mae_test))


Mean Absolute Error- Validation Baseline: 0.8312646762831265
Mean Absolute Error- Test Baseline: 0.8202247191011236


In [130]:
val = X_test_final.copy()
val["abs_error"] = (y_test_final-test_predictions).abs()
val["base_error"] = (y_test_final - X_test_final["promised_days"]).abs()
print(val.groupby("shipment_mode")[["abs_error", "base_error"]].mean())
print(val.groupby("service_level")[["abs_error", "base_error"]].mean())
print(val.groupby("carrier")[["abs_error", "base_error"]].mean())

               abs_error  base_error
shipment_mode                       
ltl_freight     0.509560    0.748894
parcel          0.544527    0.832971
                abs_error  base_error
service_level                        
Expedited        0.518841    0.806876
Ground           0.544515    0.844524
LTL Guaranteed   0.506964    0.672646
LTL Standard     0.510411    0.773862
Overnight        0.570496    0.798479
TwoDay           0.560043    0.848485
                  abs_error  base_error
carrier                                
Cobalt Regional    0.426409    0.745856
Falcon Post        0.656241    0.870825
Ironclad Freight   0.540092    0.685328
Northstar Parcel   0.496695    0.638922
Summit Express     0.436853    1.057167
Vela Freight       0.468589    0.834197


In [131]:
df.columns

Index(['shipment_id', 'order_date', 'ship_date', 'ship_dow', 'ship_month',
       'shipment_mode', 'carrier', 'service_level', 'origin_metro',
       'dest_metro', 'origin_zip', 'dest_zip', 'origin_lat', 'origin_lon',
       'dest_lat', 'dest_lon', 'distance_miles', 'zone', 'weight_lbs',
       'length_in', 'width_in', 'height_in', 'pallet_count',
       'residential_flag', 'signature_required', 'declared_value_usd',
       'fuel_surcharge_pct', 'weather_severity', 'carrier_congestion_index',
       'promised_days', 'carrier_cost_usd', 'transit_days',
       'delivered_on_time', 'weight_lbs_na', 'length_in_na', 'width_in_na',
       'height_in_na'],
      dtype='str')

In [132]:
df_test= pd.read_csv("../data/test_engineered_features.csv")


In [133]:
for col in ["carrier", "service_level", "origin_metro", "dest_metro", "shipment_mode"]:
    df_test[col] = pd.Categorical(df_test[col], categories=X[col].cat.categories)

In [134]:
X_test_new = df_test[X.columns]
print(X_test_new.columns)
test_predictions = model.predict(X_test_new)

Index(['ship_dow', 'ship_month', 'shipment_mode', 'carrier', 'service_level',
       'origin_metro', 'dest_metro', 'origin_lat', 'origin_lon', 'dest_lat',
       'dest_lon', 'distance_miles', 'zone', 'weight_lbs', 'length_in',
       'width_in', 'height_in', 'pallet_count', 'residential_flag',
       'signature_required', 'declared_value_usd', 'fuel_surcharge_pct',
       'weather_severity', 'carrier_congestion_index', 'promised_days',
       'carrier_cost_usd', 'weight_lbs_na', 'length_in_na', 'width_in_na',
       'height_in_na'],
      dtype='str')
